# ⚾ MLB Analytics Pipeline
### Phase 1 — Data Collection

This notebook:
1. Installs all required packages
2. Optionally mounts Google Drive (so data persists after session ends)
3. Runs the full pipeline — schedule, stats, park factors, Statcast
4. Lets you explore matchups interactively

**Run cells top to bottom on first use. After that, only run Cell 8 daily.**

In [ ]:
# ── CELL 1: Install packages (run once, then skip) ──────────────────────────
!pip install pybaseball mlb-statsapi pandas numpy requests sqlalchemy -q
print('✓ Packages installed')

In [ ]:
# ── CELL 2: Mount Google Drive (OPTIONAL but recommended) ───────────────────
# Your database will persist between sessions.
# Skip this cell if you don't care about persistence.

MOUNT_DRIVE = True  # Set to False to skip

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.makedirs('/content/drive/MyDrive/mlb_analytics', exist_ok=True)
    print('✓ Google Drive mounted')
    print('  Database will be saved to: /content/drive/MyDrive/mlb_analytics/mlb_analytics.db')
else:
    print('Skipping Drive mount — data will be lost when session ends')

In [ ]:
# ── CELL 3: Pull pipeline code from GitHub ──────────────────────────────────
# Replace YOUR_GITHUB_USERNAME with your actual username after you push to GitHub

# Option A: Pull from GitHub (recommended once you've pushed)
# !git clone https://github.com/YOUR_GITHUB_USERNAME/mlb-analytics.git
# import sys; sys.path.insert(0, 'mlb-analytics')

# Option B: Upload fetch_data.py directly to Colab and import it
# from google.colab import files
# files.upload()  # Upload fetch_data.py when prompted

# Option C: Paste the pipeline inline (copy from fetch_data.py)
# The cells below duplicate the key functions so you can run without GitHub

print('Choose one of the options above, or run cells 4+ which include inline code')

In [ ]:
# ── CELL 4: Config + Imports ─────────────────────────────────────────────────
import pandas as pd
import numpy as np
import sqlite3
import os
from datetime import datetime, date, timedelta
import statsapi
from pybaseball import (
    statcast, batting_stats, pitching_stats,
    park_factors, playerid_lookup, cache
)
cache.enable()

CURRENT_SEASON = datetime.now().year

# Change this path if you mounted Google Drive in Cell 2
DB_PATH = '/content/drive/MyDrive/mlb_analytics/mlb_analytics.db' if MOUNT_DRIVE else 'mlb_analytics.db'

print(f'Season: {CURRENT_SEASON}')
print(f'DB path: {DB_PATH}')

In [ ]:
# ── CELL 5: Import pipeline module ───────────────────────────────────────────
# If you cloned the repo in Cell 3, use this:
# from pipeline.fetch_data import *

# Otherwise, copy-paste the content of fetch_data.py here
# (or upload the file and use the import above)

# For now, we'll import the key functions assuming fetch_data.py is in the path
import sys
sys.path.insert(0, '/content/mlb-analytics')  # Adjust if your clone path differs

try:
    from pipeline.fetch_data import (
        init_db, get_conn,
        fetch_schedule,
        fetch_batting_stats,
        fetch_pitching_stats,
        fetch_park_factors_data,
        fetch_statcast_data,
        build_todays_matchup_preview,
        analyze_batter_vs_pitch_type,
        nightly_update,
        run_full_pipeline,
    )
    print('✓ Pipeline module loaded')
except ImportError:
    print('Module not found — make sure you cloned the repo or uploaded fetch_data.py')
    print('Run: !git clone https://github.com/YOUR_USERNAME/mlb-analytics.git')

In [ ]:
# ── CELL 6: FIRST TIME ONLY — Run full pipeline ───────────────────────────────
# This takes ~15-20 minutes on the first run.
# statcast_days=30 pulls 30 days of pitch data (~500k rows)
# Increase to 90 for richer model training data (slower)

run_full_pipeline(statcast_days=30)

In [ ]:
# ── CELL 7: Explore — today's matchup preview ────────────────────────────────
matchups = build_todays_matchup_preview()
matchups

In [ ]:
# ── CELL 8: DAILY USE — Nightly update (run every morning) ───────────────────
# Only fetches today's schedule + last 2 days of Statcast. Fast (~2 min).

nightly_update()

In [ ]:
# ── CELL 9: Explore — batter vs pitch type ────────────────────────────────────
# Change the names below to whoever is pitching today

batter  = 'Aaron Judge'      # Change me
pitcher = 'Gerrit Cole'      # Change me (or leave blank for all pitchers)

matchup_df = analyze_batter_vs_pitch_type(batter, pitcher)
matchup_df

In [ ]:
# ── CELL 10: Explore — park factor table ─────────────────────────────────────
conn = get_conn()
parks = pd.read_sql('SELECT * FROM park_factors ORDER BY hr_factor DESC', conn)
conn.close()

print('Park factors — sorted by HR factor (100 = neutral):')
parks

In [ ]:
# ── CELL 11: Explore — top batters by xwOBA ─────────────────────────────────
conn = get_conn()
batters = pd.read_sql("""
    SELECT name, team, pa, avg, obp, slg, woba, xwoba, barrel_pct, k_pct, bb_pct
    FROM batting_stats
    ORDER BY xwoba DESC
    LIMIT 20
""", conn)
conn.close()

print('Top 20 batters by xwOBA:')
batters

In [ ]:
# ── CELL 12: Explore — top pitchers by FIP ──────────────────────────────────
conn = get_conn()
pitchers = pd.read_sql("""
    SELECT name, team, ip, era, fip, xfip, siera, k_pct, bb_pct, k_bb_pct, stuff_plus
    FROM pitching_stats
    WHERE ip >= 30
    ORDER BY fip ASC
    LIMIT 20
""", conn)
conn.close()

print('Top 20 qualified starters by FIP (lower = better):')
pitchers

In [ ]:
# ── CELL 13: Verify database ─────────────────────────────────────────────────
conn = get_conn()
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

print('Tables in database:')
for t in tables['name']:
    count = pd.read_sql(f'SELECT COUNT(*) as n FROM {t}', conn).iloc[0]['n']
    print(f'  {t:<25s}  {count:>10,} rows')

conn.close()